# 04 OASIS and runoff-mapper: coupler restart and river routing

OASIS and the runoff-mapper each have a small step, so they share this notebook:

* `oasis` (`EoceneOASIS`) rewrites the OASIS restart `rstos.nc`: no sea ice and a constant SST
  (`eocene.oasis.constant_sst`, 290 K by default);
* `rnfm` (`EoceneRNFM`) rebuilds the runoff-mapper basins (`runoff_maps.nc`) from the Herold et al. (2014) river
  flow directions.

**Run `01_quickstart.ipynb` first**: this notebook reads its configuration and output.

In [ ]:
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

from ecefast.core.config import load_config
from ecefast.core.workspace import Context

# Working directory of the tutorials: 01_quickstart.ipynb writes the config and the outputs here
WORK = Path(os.environ.get("ECEFAST_TUTORIAL_DIR", Path(tempfile.gettempdir()) / "ecefast-tutorial"))
CONFIG = WORK / "laptop.yml"

# ctx expands path templates of the Eocene period, such as "{input}/oasis/{grids.nemo.name}/rstos.nc"
ctx = Context(load_config(str(CONFIG)), "eocene")
import numpy as np

from ecefast.periods.eocene.oasis.eoceneOASIS import EoceneOASIS
from ecefast.periods.eocene.plots.helper_functions import plot_before_after
from ecefast.periods.eocene.rnfm.eoceneRNFM import EoceneRNFM

## OASIS

In [ ]:
for kind in ("needs", "reads", "writes"):
    print(f"{kind}:")
    for template in getattr(EoceneOASIS, kind):
        print("   ", ctx.expand(template) if kind != "needs" else template)

The fields that the ocean sends to the atmosphere at the first coupling step, on the PALEORCA2 grid (index
space): SST and sea-ice fraction.

In [ ]:
before = xr.open_dataset(ctx.expand("{input}/initial_state/oasis/{grids.nemo.name}/default/rstos.nc"))
after = xr.open_dataset(ctx.expand("{output}/initial_state/oasis/{grids.nemo.name}/default/rstos.nc"))
# points without ocean data hold 0 K in the present-day file: the colour scale starts at 260 K
sst = "O_SSTSST: sea surface temperature (K)"
plot_before_after(before["O_SSTSST"], after["O_SSTSST"], sst, cmap="RdYlBu_r", vmin=260, vmax=310)
plot_before_after(before["OIceFrc"], after["OIceFrc"], "OIceFrc: sea-ice fraction", cmap="Blues_r");

In [ ]:
print(f"constant_sst = {ctx.config.eocene.oasis.constant_sst} K")
for var in ["O_SSTSST", "OIceFrc", "OIceTck", "OSnwTck", "O_AlbIce", "O_TepIce"]:
    print(
        f"{var:9s} present day {float(before[var].min()):8.2f} .. {float(before[var].max()):8.2f}   "
        f"Eocene {float(after[var].min()):8.2f} .. {float(after[var].max()):8.2f}"
    )

## Runoff-mapper

In [ ]:
for kind in ("needs", "reads", "writes"):
    print(f"{kind}:")
    for template in getattr(EoceneRNFM, kind):
        print("   ", ctx.expand(template) if kind != "needs" else template)

The input: the river flow direction of Herold et al. (2014) on a 1° grid (RTM convention, 1-8 = direction,
0 = ocean).

In [ ]:
flow = xr.open_dataset(ctx.expand("{herold}/herold_etal_eocene_runoff_1x1.nc"))["RTM_FLOW_DIRECTION"]
flow.plot(figsize=(8, 4), cmap="tab10")
plt.title("Herold et al. (2014) river flow direction");

The step follows the flow directions to the coast, merges small basins, and writes the result on the
runoff-mapper grid (`runoff_maps.nc`):

* `drainage_basin_id`: the basin of each land point (one colour per basin; colours repeat);
* `arrival_point_id`: the ocean points where each basin discharges (basin id; -2 elsewhere).

In [ ]:
before = xr.open_dataset(ctx.expand("{input}/runoff-mapper/runoff_maps.nc"))
after = xr.open_dataset(ctx.expand("{output}/runoff-mapper/runoff_maps.nc"))


def basins(ds):
    """Basin ids folded onto 20 colours; points outside every basin are blank."""
    ids = ds["drainage_basin_id"]
    return (ids % 20).where(ids > 0)


plot_before_after(basins(before), basins(after), "drainage_basin_id", cmap="tab20", add_colorbar=False)
plot_before_after(
    xr.where(before["arrival_point_id"] > 0, 1, np.nan),
    xr.where(after["arrival_point_id"] > 0, 1, np.nan),
    "Arrival points (arrival_point_id > 0)",
    cmap="Reds",
    vmin=0,
    vmax=1,
    add_colorbar=False,
);

In [ ]:
for name, ds in [("present day", before), ("Eocene", after)]:
    ids = ds["drainage_basin_id"].values
    print(f"{name:12s} basins: {len(np.unique(ids[ids > 0]))}")

Without ice sheets in the Eocene, `calving_point_id` is set equal to `arrival_point_id` (the step logs a warning
about it).